## LEAD MANAGEMENT REPORT

In [1]:
import pandas as pd
import numpy as np
from gspread_pandas import Spread, conf

In [2]:
# --- 1. CONFIGURATION & CONNECTION ---
config_path = r'C:\Users\USER\Downloads\Emarath_global\service_account.json'
emarath_global_sheet_url = "https://docs.google.com/spreadsheets/d/1ztfkIIEeL9EmSdLQIGhBGIFbraMLmqEBwc8FP-CA_3c/edit?gid=822921427#gid=822921427"


In [3]:
# Set your target date here
target_date_str = "07/03/2026" 
target_date_dt = pd.to_datetime(target_date_str, dayfirst=True)

target_bde_sheets = [
    'RAHIB', 'HABIYA', 'BURHANA', 'SHAMNA', 'ARUN',
    'CHAITHANYA', 'ZAKIYA', 'SAFAN', 'SUSHANTHIKA', 
    'ADWAITHA', 'NEHA', 'GOWTHAM', 'AMINA', 'RINSIYA', 'ARSHAD',  'NAFI',
    "SHIHAD", "RAHIYAD",
    "SHYAMJIL", 'ADHIL', "HIBA", 'NAJA', 'AJIN', 'SHABNA', 'FARSANA',
]

lead_statuses = ["WON", "SUPER HOT", "HOT", "WARM", "COLD", "BOOKING", "WHATS APP ENGAGE"]

c = conf.get_config(file_name=config_path)
emarath_spread = Spread(emarath_global_sheet_url, config=c)

all_data_list = []

for sheet_name in target_bde_sheets:
    try:
        df = emarath_spread.sheet_to_df(sheet=sheet_name, index=None, header_rows=1)
        
        mapping_cols = {
            'AGENT'       : 'AGENT',
            'COUNTRY'     : 'REGION',
            'CUSTOMER PATH': 'CUSTOMER PATH',
            'PRODUCT 1'   : 'PRODUCT',
            'NAME'        : 'NAME',
            'PHONE NO 1'  : 'PHONE NO',
            'STATUS'      : 'STATUS',
            'DATE'        : 'DATE',
            'CALL STATUS' : 'CALL STATUS'
        }
        
        if not df.empty and all(col in df.columns for col in mapping_cols.keys()):
            subset = df[list(mapping_cols.keys())].copy()
            subset.rename(columns=mapping_cols, inplace=True)

            # --- CONDITION 1: Must be on target date ---
            subset['DATE'] = pd.to_datetime(subset['DATE'], errors='coerce', dayfirst=True)
            subset = subset[subset['DATE'].dt.date == target_date_dt.date()]

            if subset.empty:
                continue

            subset['CUSTOMER PATH'] = subset['CUSTOMER PATH'].astype(str).str.strip().str.upper()
            subset = subset[subset['CUSTOMER PATH'] == 'LEAD']
            subset = subset[~subset['PHONE NO'].astype(str).str.strip().str.upper().isin(['', 'NAN', 'NONE', 'N/A', 'UNKNOWN', '0'])]
            
            if subset.empty:
                continue

            # Clean remaining string columns
            for col in [ 'PHONE NO', 'CUSTOMER PATH', 'CALL STATUS']:
                subset[col] = subset[col].astype(str).str.strip().str.upper()

            all_data_list.append(subset)
                
    except Exception as e:
        print(f"Error in sheet {sheet_name}: {e}")

# --- PROCESSING & PIVOTING ---
if all_data_list:
    master_df = pd.concat(all_data_list, ignore_index=True)
    master_df['is_unattended'] = (
        (master_df['CALL STATUS'] == '') | 
        (master_df['CALL STATUS'] == 'NAN')
    )

    # Total landed leads
    total_leads = (
        master_df
        .groupby('AGENT')['PHONE NO']
        .count()
        .to_frame('TOTAL LANDED LEADS')
    )

    unattended_counts = (
        master_df[master_df['is_unattended']]
        .groupby('AGENT')
        .size()
        .to_frame('UNATTENDED')
    )

    # Status pivot
    status_pivot = master_df.pivot_table(
        index='AGENT',
        columns='STATUS',
        aggfunc='size',
        fill_value=0
    )
    status_pivot = status_pivot.add_suffix('_S')

    # Join
    final_report = (
        total_leads
        .join(unattended_counts, how='left')
        .join(status_pivot, how='left')
        .fillna(0)
        .astype(int)
    )


    ordered_cols = ['TOTAL LANDED LEADS'] + [s + '_S' for s in lead_statuses] + ['UNATTENDED']
    valid_cols = [c for c in ordered_cols if c in final_report.columns]
    final_report = final_report.reindex(columns=valid_cols, fill_value=0)


    final_report.columns = [
        col.replace('_S', '') if col.endswith('_S') else col
        for col in final_report.columns
    ]

    final_report = final_report.sort_index()
    final_report.loc['TOTAL'] = final_report.sum()

    # --- EXPORT ---
    file_path = f"./Output_Data/Lead_Management_Report_{target_date_dt.strftime('%d-%m-%Y')}.xlsx"
    final_report.to_excel(file_path, merge_cells=False)

    print(f"Report for {target_date_dt.date()} generated successfully")
    print(f"Saved to: {file_path}")
    print(f"\nTotal valid leads processed: {master_df.shape[0]}")
    # print(final_report)

else:
    print(f"No valid records found for {target_date_dt.date()}.")

Report for 2026-03-07 generated successfully
Saved to: ./Output_Data/Lead_Management_Report_07-03-2026.xlsx

Total valid leads processed: 312


In [6]:
# Set your target date here
days_back = 31
today = pd.Timestamp.now().normalize()
start_date = today - pd.Timedelta(days=days_back)



target_bde_sheets = [
    'RAHIB', 'HABIYA', 'BURHANA', 'SHAMNA', 'ARUN', 
    'CHAITHANYA', 'ZAKIYA',  'SAFAN', 'SUSHANTHIKA', 
    'ADWAITHA', 'NEHA', 'GOWTHAM', 'AMINA',  
    'RINSIYA', 'ARSHAD','NAFI'
]

lead_statuses = ["WON", "SUPER HOT", "HOT", "WARM", "COLD", "BOOKING", "WHATS APP ENGAGE"]

c = conf.get_config(file_name=config_path)
emarath_spread = Spread(emarath_global_sheet_url, config=c)

all_data_list = []

for sheet_name in target_bde_sheets:
    try:
        df = emarath_spread.sheet_to_df(sheet=sheet_name, index=None, header_rows=1)
        
        mapping_cols = {
            'AGENT'       : 'AGENT',
            'COUNTRY'     : 'REGION',
            'CUSTOMER PATH': 'CUSTOMER PATH',
            'PRODUCT 1'   : 'PRODUCT',
            'NAME'        : 'NAME',
            'PHONE NO 1'  : 'PHONE NO',
            'STATUS'      : 'STATUS',
            'DATE'        : 'DATE',
            'CALL STATUS' : 'CALL STATUS'
        }
        
        if not df.empty and all(col in df.columns for col in mapping_cols.keys()):
            subset = df[list(mapping_cols.keys())].copy()
            subset.rename(columns=mapping_cols, inplace=True)

            # --- CONDITION 1: Must be on target date ---
            subset['DATE'] = pd.to_datetime(subset['DATE'], errors='coerce', dayfirst=True)
            # subset = subset[(subset['DATE'] >= start_date) & (subset['DATE'] <= today)]

            # subset['DATE'] = pd.to_datetime(subset['DATE'], errors='coerce', dayfirst=True)
            # subset = subset[subset['DATE'].dt.date == target_date_dt.date()]

            if subset.empty:
                continue

            subset['CUSTOMER PATH'] = subset['CUSTOMER PATH'].astype(str).str.strip().str.upper()
            subset = subset[subset['CUSTOMER PATH'] == 'LEAD']
            subset = subset[~subset['PHONE NO'].astype(str).str.strip().str.upper().isin(['', 'NAN', 'NONE', 'N/A', 'UNKNOWN', '0'])]


            if subset.empty:
                continue

            # Clean remaining string columns
            for col in [ 'PHONE NO', 'CUSTOMER PATH', 'CALL STATUS']:
                subset[col] = subset[col].astype(str).str.strip().str.upper()

            all_data_list.append(subset)
                
    except Exception as e:
        print(f"Error in sheet {sheet_name}: {e}")

# --- PROCESSING & PIVOTING ---
if all_data_list:
    master_df = pd.concat(all_data_list, ignore_index=True)
    master_df['is_unattended'] = (
        (master_df['CALL STATUS'] == '') | 
        (master_df['CALL STATUS'] == 'NAN')
    )

    # Total landed leads
    total_leads = (
        master_df
        .groupby('AGENT')['PHONE NO']
        .count()
        .to_frame('TOTAL LANDED LEADS')
    )

    unattended_counts = (
        master_df[master_df['is_unattended']]
        .groupby('AGENT')
        .size()
        .to_frame('UNATTENDED')
    )

    # Status pivot
    status_pivot = master_df.pivot_table(
        index='AGENT',
        columns='STATUS',
        aggfunc='size',
        fill_value=0
    )
    status_pivot = status_pivot.add_suffix('_S')

    # Join
    final_report = (
        total_leads
        .join(unattended_counts, how='left')
        .join(status_pivot, how='left')
        .fillna(0)
        .astype(int)
    )


    ordered_cols = ['TOTAL LANDED LEADS'] + [s + '_S' for s in lead_statuses] + ['UNATTENDED']
    valid_cols = [c for c in ordered_cols if c in final_report.columns]
    final_report = final_report.reindex(columns=valid_cols, fill_value=0)


    final_report.columns = [
        col.replace('_S', '') if col.endswith('_S') else col
        for col in final_report.columns
    ]

    final_report = final_report.sort_index()
    final_report.loc['TOTAL'] = final_report.sum()

    # --- EXPORT ---
    date_range_str = f"{start_date.strftime('%d%m')}_to_{today.strftime('%d%m')}"
    file_path = f"./Output_Data/Lead_Management_Report_{date_range_str}.xlsx"
    # file_path = f"./Output_Data/Lead_Management_Report_{target_date_dt.strftime('%d-%m-%Y')}.xlsx"
    final_report.to_excel(file_path, merge_cells=False)

    # print(f"Report for {date_range_str.date()} generated successfully")
    print(f"Saved to: {file_path}")
    print(f"\nTotal valid leads processed: {master_df.shape[0]}")
    # print(final_report)

else:
    print(f"No valid records found for {file_path}.")

Saved to: ./Output_Data/Lead_Management_Report_3001_to_0203.xlsx

Total valid leads processed: 5880


## DATA OF OTHER LANGUAGE 

In [ ]:
TARGET_DATE_STR = "27/02/2026" 
target_date_dt = pd.to_datetime(TARGET_DATE_STR, dayfirst=True)

TARGET_BDE_SHEETS = [
    'RAHIB', 'HABIYA', 'BURHANA', 'SHAMNA', 'ARUN', 
    'CHAITHANYA', 'ZAKIYA', 'AJIN', 'SAFAN', 'SUSHANTHIKA', 
    'ADWAITHA', 'NEHA', 'GOWTHAM', 'AMINA', 'ADHIL', 
    'SHABNA', 'FARSANA', 'RINSIYA', 'ARSHAD','NAFI'
]

# --- INITIALIZE ---
try:
    c = conf.get_config(file_name=config_path)
    emarath_spread = Spread(emarath_global_sheet_url, config=c)
except Exception as e:
    print(f"Connection Error: {e}")
    exit()

all_data_list = []

print(f"🔍 Searching for Non-Malayalam leads on {TARGET_DATE_STR}...")

for sheet_name in TARGET_BDE_SHEETS:
    try:
        # Read sheet
        df = emarath_spread.sheet_to_df(sheet=sheet_name, index=None, header_rows=1)
        
        # Define columns to pull (Make sure 'LANGUAGE' matches your sheet header exactly)
        mapping_cols = {
            'NAME': 'NAME',
            'PHONE NO 1': 'PHONE NO',
            'LANGUAGE': 'LANGUAGE',
            'AGENT': 'AGENT',
            'DATE': 'DATE',
            'CUSTOMER PATH': 'CUSTOMER PATH'
        }
        
        # Check if sheet has required columns
        if not df.empty and all(col in df.columns for col in mapping_cols.keys()):
            subset = df[list(mapping_cols.keys())].copy()
            subset.rename(columns=mapping_cols, inplace=True)

            # 1. Filter by Date
            subset['DATE'] = pd.to_datetime(subset['DATE'], errors='coerce', dayfirst=True)
            subset = subset[subset['DATE'].dt.date == target_date_dt.date()]

            # 2. Filter for 'LEAD' path only
            subset['CUSTOMER PATH'] = subset['CUSTOMER PATH'].astype(str).str.strip().str.upper()
            subset = subset[subset['CUSTOMER PATH'] == 'LEAD']
            subset = subset[~subset['PHONE NO'].astype(str).str.strip().str.upper().isin(['', 'NAN', 'NONE', 'N/A', 'UNKNOWN', '0'])]

            if subset.empty:
                continue

            all_data_list.append(subset)
                
    except Exception as e:
        print(f"Error reading {sheet_name}: {e}")

# --- FILTERING ---
if all_data_list:
    master_df = pd.concat(all_data_list, ignore_index=True)
    lang_clean = master_df['LANGUAGE'].astype(str).str.strip().str.upper()
    is_not_blank = (lang_clean != '') & (lang_clean != 'NAN') & (master_df['LANGUAGE'].notnull())
    is_not_malayalam = ~lang_clean.str.contains('MALAYALAM', na=False)
    non_malayalam_list = master_df[is_not_blank & is_not_malayalam].copy()
    final_output = non_malayalam_list[['NAME', 'PHONE NO', 'LANGUAGE', 'AGENT']]

    file_name = f"./Output_Data/Non_Malayalam_Leads_{target_date_dt.strftime('%d-%m-%Y')}.xlsx"
    final_output.to_excel(file_name, index=False)
    print(f"✅ SUCCESS!")
    print(f"Found {len(final_output)} valid non-Malayalam leads (Blanks removed).")

else:
    print(f"No records found for the date {TARGET_DATE_STR}.")